# CNN to classify Cifar-10 dataset (Images)



So far, we saw how to build a Dense Neural Network (DNN) that classified images of digits (MNIST) or even fashion images (Fashion-MNIST). Here we will instead, recognize the 10 classes of CIFAR ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship' and 'truck'). There are some key differences between these two image datasets that we need to take into account.

First, while MNIST were 28x28 monochrome images (1 color channel), CIFAR is 32x32 color images (3 color channels).

Second, MNIST images are simple, containing just the object centered in the image, with no background. Conversely, CIFAR ones are not centered and can have the object with a background, such as airplanes that might have a cloudy sky behind them! Those differences are the main reason to use a CNN instead of a DNN.

## Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten
from tensorflow.keras.callbacks import EarlyStopping

This cell imports all the necessary libraries for building, training, and evaluating a Convolutional Neural Network (CNN) using TensorFlow and Keras.
- `numpy` is used for numerical operations, especially with arrays.
- `matplotlib.pyplot` is for plotting and visualizing data, such as images and training history.
- `tensorflow` is the core machine learning library.
- `Sequential` from `tensorflow.keras.models` is used to create a linear stack of layers for the neural network.
- `Dense`, `Conv2D`, `MaxPool2D`, and `Flatten` from `tensorflow.keras.layers` are the building blocks of our CNN:
    - `Conv2D`: Defines 2D convolutional layers, essential for image processing.
    - `MaxPool2D`: Implements 2D max pooling layers to reduce dimensionality.
    - `Flatten`: Flattens the output of convolutional layers into a 1D array for the dense layers.
    - `Dense`: Represents fully connected layers in the network.
- `EarlyStopping` from `tensorflow.keras.callbacks` is used to prevent overfitting by stopping training when a monitored metric (e.g., validation loss) stops improving.

## Import and Inspect Dataset

This cell loads the CIFAR-10 dataset, which is a common benchmark dataset in computer vision. It consists of 60,000 32x32 color images in 10 classes, with 6,000 images per class. The dataset is split into 50,000 training images and 10,000 testing images.
- `tf.keras.datasets.cifar10` provides convenient access to the dataset.
- `load_data()` downloads and loads the images and their corresponding labels, separating them into training and testing sets.

Cifar-10 repository: https://www.cs.toronto.edu/~kriz/cifar.html






In [ ]:
cifar10 = tf.keras.datasets.cifar10
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

This cell prints the shapes of the `train_images`, `train_labels`, `test_images`, and `test_labels` arrays. Understanding the shape helps in knowing the dimensions of the data, such as the number of images, image height, width, and color channels, as well as the structure of the labels.

- The image data shape is: `(#images, img_heigth, img_width, #channels)`, where channels are in RGB format (red, green, blue).
- The labels shape is `(#images, label)`, where label goes from 0 to 9.


In [ ]:
print(train_images.shape, train_labels.shape)
print(test_images.shape, test_labels.shape)

This cell displays the raw pixel values of the first image in the `train_images` dataset. Each pixel is represented by a set of three numbers (RGB values), ranging from 0 to 255.

In [ ]:
train_images[0]

This cell uses `matplotlib.pyplot.imshow()` to render the first training image visually. It provides a quick way to see what the raw pixel data actually represents.

In [ ]:
plt.imshow(train_images[0]);

In [ ]:
train_labels[1][0]

    The CIFAR labels happen to be arrays, which is why you need the extra index

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
class_names[9] # The List's index is the label

In [ ]:
idx = train_labels[1][0]
class_names[idx]

In [ ]:
print("\t", class_names[train_labels[1][0]])
plt.imshow(train_images[1])
plt.axis('off');

In [ ]:
 def plot_train_img(img, size=2):
    label = train_labels[img][0]
    plt.figure(figsize=(size,size))
    print("Label {} - {}".format(label, class_names[label]))
    plt.imshow(train_images[img])
    plt.axis('off')
    plt.show()

In [ ]:
plot_train_img(1)

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

Note that images are in color, not centered and with different backgrounds

## Preprocessing dataset

This cell checks the maximum pixel value in the `test_images` dataset before normalization. Image pixel values typically range from 0 to 255.

In [ ]:
test_images.max()

This cell normalizes the pixel values of both `train_images` and `test_images`. By dividing all pixel values by 255.0, the pixel values are scaled to a range between 0 and 1. This is a common preprocessing step in neural networks, as it helps in faster convergence during training and can improve model performance.

In [ ]:
# Normalize pixel values to be between 0 and 1
train_images = train_images / 255.0
test_images = test_images / 255.0

After normalization, this cell re-checks the maximum pixel value in `test_images`. We expect the output to be 1.0, confirming that the normalization was successful.

In [ ]:
test_images.max()

In [ ]:
plt.hist(train_labels[:5_000]);

In [ ]:
val_images = train_images[:5_000]
val_labels = train_labels[:5_000]
print(val_images.shape, val_labels.shape)

In [ ]:
train_images = train_images[5_000:]
train_labels = train_labels[5_000:]
print(train_images.shape, train_labels.shape)

This cell plots histograms for the training, validation, and test labels on the same graph. This visualization allows for a quick comparison of the class distributions across all three datasets, ensuring that each set has a relatively similar representation of classes.

In [ ]:
plt.hist(train_labels, alpha=0.5)
plt.hist(val_labels, alpha=0.5)
plt.hist(test_labels, alpha=0.5);

## Create Model Architecture and Compile

On [Convolution layers](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D),
- strides is an integer or tuple/list of 2 integers, specifying the strides of the convolution along the height and width. Default (1,1).
- padding: one of "valid" or "same" (case-insensitive). Default = 'valid'.
  - "valid" means no padding.  
  - "same" results in padding with zeros evenly
to the left/right or up/down of the input such that output has the same


In [ ]:
model = Sequential()


model.add(Conv2D(
    filters=32,
    kernel_size=(3,3),
    activation='relu',
    input_shape=(32, 32, 3))
)
model.add(MaxPool2D(2, 2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPool2D())

model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(10, activation='softmax'))

model.summary()

In [ ]:
LOSS = 'sparse_categorical_crossentropy'
OPTIMIZER = 'adam'

# Compile the model
model.compile(optimizer=OPTIMIZER,
              loss=LOSS,
              metrics=['accuracy'])

## Training

In [ ]:
NUM_EPOCHS = 20

early_stop = EarlyStopping(monitor='val_loss',patience=3)

In [ ]:
# Fit the model
history = model.fit(train_images,
                    train_labels,
                    epochs=NUM_EPOCHS,
                    validation_data=(val_images, val_labels),
                    callbacks=[early_stop]
)

In [ ]:
# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
#plt.xlim([0,NUM_EPOCHS])
plt.ylim([0.4,1.0])
plt.show()

## Evaluate Model

This cell evaluates the trained model's performance on the `test_images` and `test_labels`. It will output the test loss and test accuracy, providing an indication of how well the model generalizes to unseen data.

In [ ]:
model.evaluate(test_images, test_labels)

**Accuracy**
- Train: 85% - 90%;
- Validation: 68%-70%
- Test: 66%-68%

In [ ]:
predictions = np.argmax(model.predict(test_images), axis=-1)
predictions.shape

This cell imports `classification_report` and `confusion_matrix` from `sklearn.metrics`. These functions are essential tools for a detailed evaluation of a classification model's performance, providing metrics like precision, recall, F1-score, and a visual representation of correct and incorrect predictions.

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
print(classification_report(test_labels, predictions, target_names=class_names))

This cell computes the `confusion_matrix`. A confusion matrix is a table that summarizes the performance of a classification model on a set of test data for which the true values are known. It shows the number of correct and incorrect predictions made by the model, broken down by each class.

In [ ]:
confusion_matrix(test_labels,predictions)

This cell simply displays the `class_names` list, which contains the human-readable labels for the 10 CIFAR-10 classes. This is useful for interpreting the confusion matrix and classification report.

In [ ]:
class_names

This cell visualizes the confusion matrix using a heatmap from the `seaborn` library. A heatmap makes it easier to visually identify patterns of correct and incorrect classifications, highlighting which classes are frequently confused with others.

In [ ]:
import seaborn as sns
plt.figure(figsize=(15,8))
sns.heatmap(confusion_matrix(test_labels,predictions), cmap='Blues', annot=True, fmt='g');

## Testing Model (Predicting)

In [ ]:
plt.imshow(test_images[15]);

In [ ]:
test_labels[15][0]

In [ ]:
class_names[8]

In [ ]:
test_images[15].shape

The input Tensor shape should be: (num_images, width, height, color_channels)

In [ ]:
my_image = test_images[15]
my_image = my_image.reshape(1,32,32,3)
my_image.shape

In [ ]:
img_pred = np.argmax(model.predict(my_image))
class_names[img_pred]

In [ ]:
img_pred

In [ ]:
pred_prob = model.predict(my_image)[0][img_pred]
pred_prob

In [ ]:
def img_pred(img, size=4):
    label = test_labels[img][0]
    my_image = test_images[img]
    plt.figure(figsize=(size,size))
    plt.imshow(my_image)
    my_image = my_image.reshape(1,32,32,3)
    img_pred = np.argmax(model.predict(my_image))
    pred_label = class_names[img_pred]
    pred_prob = model.predict(my_image)[0][img_pred]
    print(" Label {} <=> Pred: {} with prob {:.2}".format(
        class_names[label],
        pred_label,
        pred_prob))
    plt.grid(False)
    plt.axis('off')
    plt.show()

In [ ]:
img_pred(0)

In [ ]:
for i in range (10):
  img_pred(i)

## Saving the model

In [ ]:
!pwd # Linux command, shows where we are in CoLab's folders

In [ ]:
model.save('cifar_10_model.keras')

Use [Netron](https://netron.app) to visualize the model, hyperparameters, tensor shapes, etc. Netron is a viewer for neural network, deep learning and machine learning models (See [GitHub](https://github.com/lutzroeder/netron) for instructions about instalation in your desktop).

# Converting to TensorFlow Lite

You can convert the TF trained model using:  `TFLiteConverter.from_keras_model(model)`

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

Alternativally, is possible to convert the model from a saved model: `tf.lite.TFLiteConverter.from_saved_model(model_cifar10)` or from a Keras saved model: `tf.lite.TFLiteConverter.from_keras_model(model_cifar10.h5)`

In [ ]:
# model_path = '/content/cifar_10_model'
# model_cifar10 = tf.keras.models.load_model(model_path)
# converter = tf.lite.TFLiteConverter.from_saved_model(model_cifar10)
# converter = tf.lite.TFLiteConverter.from_keras_model(model_cifar10)

In [ ]:
tflite_model = converter.convert()

In [ ]:
# Save .tflite model
open("/content/cifar10.tflite","wb").write(tflite_model)

## Dynamic range quantization
The simplest form of post-training quantization statically quantizes only the weights from floating point to integer, which has 8-bits of precision:

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()


In [ ]:
# Save .tflite model
open("/content/cifar10_quant.tflite","wb").write(tflite_quant_model)

Use [Netron](https://netron.app) to visualize the quantized model. Pay attention that now the weights are int8.

## Uploading a saved TF model for convertion

If you'd like to upload your previous saved model in order to avoid re-train it:

1.   On the left of the UI click on the folder icon and go to sub-folder "content"
2.   Click on the three dots to the right of folder `content` file and select upload




In [ ]:
model_path = '/content/cifar_10_model.keras'
model_cifar10 = tf.keras.models.load_model(model_path)
converter = tf.lite.TFLiteConverter.from_keras_model(model_cifar10)

In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

In [ ]:
# Save .tflite model
tflite_model_size = open("/content/cifar10_quant_model.tflite","wb").write(tflite_quant_model)
print("Quantized model (DEFAULT) is {:,} bytes".format(tflite_model_size))

## Testing TFLite Model

In [ ]:
interpreter = tf.lite.Interpreter("/content/cifar10_quant_model.tflite")

In [ ]:
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [ ]:
input_details

In [ ]:
output_details

In [ ]:
def set_input_tensor(interpreter, image):
    tensor_index = interpreter.get_input_details()[0]['index']
    input_tensor = interpreter.tensor(tensor_index)()[0]
    input_tensor[:, :] = image

In [ ]:
image = test_images[0]
plt.imshow(image);

In [ ]:
set_input_tensor(interpreter, image)
interpreter.invoke()
output_details = interpreter.get_output_details()[0]

In [ ]:
interpreter.get_tensor(output_details['index'])

In [ ]:
np.squeeze(interpreter.get_tensor(output_details['index']))

In [ ]:
output = np.squeeze(interpreter.get_tensor(output_details['index']))
output

In [ ]:
img_pred = np.argmax(output)
class_names[img_pred]

In [ ]:
img_pred

In [ ]:
output[img_pred]

In [ ]:
def classify_image(image):
    set_input_tensor(interpreter, image)
    interpreter.invoke()
    output_details = interpreter.get_output_details()[0]
    output = np.squeeze(interpreter.get_tensor(output_details['index']))
    img_pred = np.argmax(output)
    pred_label = class_names[img_pred]
    pred_prob = output[img_pred]
    plt.imshow(image)
    print(" Pred: {} with prob {:.2}".format(
        pred_label,
        pred_prob))
    plt.grid(False)
    plt.axis('off')
    plt.show()

In [ ]:
classify_image(test_images[0])

In [ ]:
for i in range (10):
  classify_image(test_images[i])

# TensorFlow Lite Micro

In [ ]:
import tensorflow as tf

In [ ]:
model_path = '/content/cifar_10_model.keras'
model_cifar10 = tf.keras.models.load_model(model_path)
converter = tf.lite.TFLiteConverter.from_keras_model(model_cifar10)

In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

In [ ]:
# Save .tflite model
tflite_model_size = open("/content/cifar10_quant_model.tflite","wb").write(tflite_quant_model)
print("Quantized model (DEFAULT) is {:,} bytes".format(tflite_model_size))

### Generate a TensorFlow Lite for Microcontrollers Model
To convert the TensorFlow Lite quantized model into a C source file that can be loaded by TensorFlow Lite for Microcontrollers on MCUs, we simply need to use the ```xxd``` tool to convert the ```.tflite``` file into a ```.cc``` file.

**Convert to a C array**

First install xxd

In [ ]:
!apt-get update && apt-get -qq install xxd

Now, convert and save the .cc converted model

In [ ]:
MODEL_TFLITE = 'cifar10_quant_model.tflite'
MODEL_TFLITE_MICRO = 'cifar10_quant_model.cc'
!xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
REPLACE_TEXT = MODEL_TFLITE.replace('/', '_').replace('.', '_')
!sed -i 's/'{REPLACE_TEXT}'/g_model/g' {MODEL_TFLITE_MICRO}

If you'd like to download your model for safekeeping:
1. On the left of the UI click on the folder icon
2. Click on the three dots to the right of the ```cifar10_quant_model.cc``` file and select download

In [ ]:
!cat {MODEL_TFLITE_MICRO}